In [1]:
from cobra.io import read_sbml_model, write_sbml_model
from cobra.core import Reaction
from cobra.flux_analysis import flux_variability_analysis
import json

In [3]:
model = read_sbml_model("models/Recon2.2_FBC_gene_names_fixed.xml")

rxn_dict = {"atp_c": -1, "h2o_c": -1, "adp_c": 1, "h_c": 1, "pi_c": 1}
ATPM = Reaction("ATPM")
model.add_reactions([ATPM])
rxn_dict_cobra = {}
for m,s in rxn_dict.items():
    m = model.metabolites.get_by_id(m)
    rxn_dict_cobra[m] = s

ATPM.add_metabolites(rxn_dict_cobra)
ATPM.lower_bound = 8.6

Set parameter Username
Set parameter LicenseID to value 2714996
Academic license - for non-commercial use only - expires 2026-09-26


In [4]:
media_dict = {}
with open("RPMI_1640_media.json") as fh:
    media_dict = json.load(fh)

for r in model.exchanges:
    if r.lower_bound == 0:
        continue
    r.lower_bound = 0

for k,v in media_dict.items():
    if k not in model.reactions:
        print(k)
        continue
    rxn = model.reactions.get_by_id(k)
    rxn.lower_bound = -v


model.reactions.EX_lac_L_e.upper_bound = 0
model.reactions.EX_for_e.upper_bound = 0
model.reactions.EX_thf_e.upper_bound = 0
model.reactions.EX_fe2_e.upper_bound = 0
model.reactions.EX_cys_L_e.upper_bound = 0
model.reactions.EX_gt1a_hs_e.upper_bound = 0
model.reactions.EX_4abut_e.upper_bound = 0
model.reactions.EX_mthgxl_e.upper_bound = 0
model.reactions.EX_bhb_e.upper_bound = 0
model.reactions.EX_ala_L_e.upper_bound = 0
model.reactions.EX_ala_D_e.upper_bound = 0


In [6]:
#ATPM.lower_bound = 1000
model.reactions.EX_o2_e.lower_bound = -20
solution = model.optimize()
print(solution.fluxes["ATPM"])
model.summary()

8.6


8.6


Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,4.39%
asn_L_e,EX_asn_L_e,0.0236,4,1.38%
chol_e,EX_chol_e,0.005,5,0.37%
glc_D_e,EX_glc_e,0.2031,6,17.85%
gln_L_e,EX_gln_L_e,0.4206,5,30.81%
glu_L_e,EX_glu_L_e,0.05,5,3.66%
gthrd_e,EX_gthrd_e,0.05,10,7.32%
his_L_e,EX_his_L_e,0.03237,6,2.85%
ile_L_e,EX_ile_L_e,0.05,6,4.39%
inost_e,EX_inost_e,0.001969,6,0.17%


In [7]:
fva_df = flux_variability_analysis(model, processes=12)

In [8]:
eps = 1e-9
mask = (fva_df['minimum'].abs() < eps) & (fva_df['maximum'].abs() < eps)
blocked_reactions = fva_df[mask].index

In [9]:
blocked_reactions_set = set(blocked_reactions)
gap_metabolites = []
for m in model.metabolites:
    m_rxns = {r.id for r in m.reactions}
    if len(m_rxns - blocked_reactions_set) == 0:
        gap_metabolites.append(m.id)

In [10]:
print(f"N rxns: {len(model.reactions)}")
print(f"N mets: {len(model.metabolites)}")

trimmed = model.copy()
trimmed.remove_reactions(blocked_reactions)
gap_metabolites = [trimmed.metabolites.get_by_id(m) for m in gap_metabolites]
trimmed.remove_metabolites(gap_metabolites)

print(f"N rxns: {len(trimmed.reactions)}")
print(f"N mets: {len(trimmed.metabolites)}")

N rxns: 7786
N mets: 5324
Read LP format model from file /tmp/tmp1b6zzv6v.lp
Reading time = 0.02 seconds
: 5324 rows, 15572 columns, 64720 nonzeros
Read LP format model from file /tmp/tmp1b6zzv6v.lp
Reading time = 0.02 seconds
: 5324 rows, 15572 columns, 64720 nonzeros
N rxns: 4786
N mets: 2570
N rxns: 4786
N mets: 2570


In [11]:
trimmed.optimize()
trimmed.summary()

write_sbml_model(trimmed, "./Recon2.2_FBC_trimmed_final.xml")

In [ ]:
test = read_sbml_model("Recon2.2_FBC_trimmed_final.xml")
test.optimize()
test.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,4.39%
asn_L_e,EX_asn_L_e,0.0236,4,1.38%
chol_e,EX_chol_e,0.005,5,0.37%
glc_D_e,EX_glc_e,0.2031,6,17.85%
gln_L_e,EX_gln_L_e,0.4206,5,30.81%
glu_L_e,EX_glu_L_e,0.05,5,3.66%
gthrd_e,EX_gthrd_e,0.05,10,7.32%
his_L_e,EX_his_L_e,0.03237,6,2.85%
ile_L_e,EX_ile_L_e,0.05,6,4.39%
inost_e,EX_inost_e,0.001969,6,0.17%


In [ ]:
test.reactions.EX_o2_e.lower_bound = 0
test.optimize()
test.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,0.83%
asn_L_e,EX_asn_L_e,0.05,4,0.55%
asp_L_e,EX_asp_L_e,0.05,4,0.55%
chol_e,EX_chol_e,0.005,5,0.07%
cys_L_e,EX_cys_L_e,0.05,3,0.41%
glc_D_e,EX_glc_e,5,6,82.94%
gln_L_e,EX_gln_L_e,0.5,5,6.91%
glu_L_e,EX_glu_L_e,0.05,5,0.69%
gly_e,EX_gly_e,0.05,2,0.28%
gthrd_e,EX_gthrd_e,0.05,10,1.38%


In [21]:
import sys
sys.path.append("modules/")
from utils import correct_ids, model_trimming

from cobra.io import validate_sbml_model
from cobra.io import read_sbml_model
from cobra.io import write_sbml_model

In [26]:
eps_tolerance = 1e-9
bound_scaling = 10.0

# In the original model I manually fixed a GPR
# GENE_ASSOCIATION: Mbar_A0818 or Mbar_A1889 )
original_model_path = "models/Recon2.2_FBC_trimmed_final.xml"
output_model_path   = "Recon2.2_FBC_trimmed_final.xml"
final_model_path   = "models/Recon2.2/Recon2.2_FBC_trimmed_final.xml"

print(f"Reading SBML model from {original_model_path}")
mb_model = read_sbml_model(original_model_path)

# Fix chemical formula syntax errors (e.g., 'C12H17O10. Na' -> 'C12H17O10Na')
print("Fixing chemical formulas...")
import re
fixed_count = 0
for metabolite in mb_model.metabolites:
    if hasattr(metabolite, 'formula') and metabolite.formula:
        original_formula = metabolite.formula
        fixed_formula = re.sub(r'\.\s*', '', original_formula)
        if original_formula != fixed_formula:
            metabolite.formula = fixed_formula
            fixed_count += 1
print(f"Fixed {fixed_count} chemical formulas")

# Write corrected model
write_sbml_model(mb_model, output_model_path)

# Validate the corrected model
print(f"Validating SBML model from {output_model_path}")
mb_model, errors = validate_sbml_model(output_model_path)

assert len(errors["SBML_FATAL"]) == 0
assert len(errors["SBML_ERROR"]) == 0
assert len(errors["SBML_SCHEMA_ERROR"]) == 0

replace_dict = {
    "_LPAREN_":"_", 
    "_RPAREN_":"", 
    "_DASH_":"_" , 
    "_unknown_":"_",
    "unknown_": ""
}

print("Correcting model IDs")
for pattern, fix in replace_dict.items():
    print(f"- Replacing \"{pattern}\" by \"{fix}\" in model's reaction/metabolite ids")
    mb_model = correct_ids(mb_model, pattern, fix)
    print(f"- ID correcting pattern {pattern}.")

print(f"Writing fixed model to {output_model_path}")
write_sbml_model(mb_model, output_model_path)

print(f"Re-reading fixed model from {output_model_path}")
mb_model = read_sbml_model(output_model_path)

print("Final check for inconsistent IDs")
total = 0
for pattern, fix in replace_dict.items():
    for r in mb_model.reactions:
        if re.search(r.id, pattern):
            total += 1
    for m in mb_model.metabolites:
        if re.search(m.id, pattern):
            total += 1

assert total == 0
print("==========================================")
print("Success!!!")

tolerance = 1e-9

Reading SBML model from models/Recon2.2_FBC_trimmed_final.xml
Fixing chemical formulas...
Fixed 2 chemical formulas
Fixing chemical formulas...
Fixed 2 chemical formulas
Validating SBML model from Recon2.2_FBC_trimmed_final.xml
Validating SBML model from Recon2.2_FBC_trimmed_final.xml
Correcting model IDs
- Replacing "_LPAREN_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 0
N of corrected metabolites ids 0
- ID correcting pattern _LPAREN_.
- Replacing "_RPAREN_" by "" in model's reaction/metabolite ids
N of corrected reaction ids 0
N of corrected metabolites ids 0
- ID correcting pattern _RPAREN_.
- Replacing "_DASH_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 1
N of corrected metabolites ids 1
Correcting model IDs
- Replacing "_LPAREN_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 0
N of corrected metabolites ids 0
- ID correcting pattern _LPAREN_.
- Replacing "_RPAREN_" by "" in model's reaction/metabolite i

In [28]:
print(f"Trimming model {mb_model.id}")

trimmed_model = model_trimming(mb_model, eps_tolerance=eps_tolerance, bound_scaling=bound_scaling)

# Set reasonable upper bounds for exchange reactions
for r in trimmed_model.exchanges:
    if r.upper_bound > 0:
        r.upper_bound = 100

# Set upper bound for biomass reaction (adjust based on your model's expected growth rate)
# You may need to identify the correct biomass reaction name
biomass_reactions = [r for r in trimmed_model.reactions if 'biomass' in r.id.lower()]
if biomass_reactions:
    print(f"Found biomass reaction(s): {[r.id for r in biomass_reactions]}")
    # Set a reasonable upper bound - adjust as needed based on your biological system
    for biomass_rxn in biomass_reactions:
        biomass_rxn.upper_bound = 1.0  # Adjust this value as appropriate
else:
    print("Warning: No biomass reaction found")

# Restore media conditions for the trimmed model
trimmed_model.reactions.EX_o2_e.lower_bound = -20

# Restore other important bounds if needed
for k, v in media_dict.items():
    if k in trimmed_model.reactions:
        rxn = trimmed_model.reactions.get_by_id(k)
        rxn.lower_bound = -v

Trimming model MODEL1603150001
Read LP format model from file /tmp/tmp3mw2_7iy.lp
Reading time = 0.01 seconds
: 2570 rows, 9572 columns, 41566 nonzeros
Reading time = 0.01 seconds
: 2570 rows, 9572 columns, 41566 nonzeros
Removing blocked 0 reactions
Removing 0 gap metabolites
Removing 261 unconnected genes
Setting bounds
Removing blocked 0 reactions
Removing 0 gap metabolites
Removing 261 unconnected genes
Setting bounds
Found biomass reaction(s): ['biomass_reaction', 'biomass_protein', 'biomass_DNA', 'biomass_RNA', 'biomass_carbohydrate', 'biomass_lipid', 'biomass_other']
Found biomass reaction(s): ['biomass_reaction', 'biomass_protein', 'biomass_DNA', 'biomass_RNA', 'biomass_carbohydrate', 'biomass_lipid', 'biomass_other']


In [29]:
print("Testing trimmed model...")

# Optimize the original model
print("\nOptimizing original model:")
original_solution = model.optimize()
print(f"Original model objective value: {original_solution.objective_value:.6f}")

# Optimize the trimmed model
print("\nOptimizing trimmed model:")
trimmed_solution = trimmed_model.optimize()
print(f"Trimmed model objective value: {trimmed_solution.objective_value:.6f}")

# Compare the solutions
diff = abs(original_solution.objective_value - trimmed_solution.objective_value)
print(f"\nDifference in objective values: {diff:.6e}")

# Assert they're similar (within tolerance)
assert diff < 1e-5, f"Solutions differ too much: {diff}"

print("\n" + "="*50)
print("Success! Trimmed model produces similar results")
print("="*50)

# Display model statistics
print(f"\n- N reactions in original model: {len(model.reactions)}")
print(f"- N reactions in trimmed model:  {len(trimmed_model.reactions)}")
print()
print(f"- N metabolites in original model: {len(model.metabolites)}")
print(f"- N metabolites in trimmed model:  {len(trimmed_model.metabolites)}")
print()
print(f"- N genes in original model: {len(model.genes)}")
print(f"- N genes in trimmed model:  {len(trimmed_model.genes)}")

# Display summary
trimmed_model.summary()

Testing trimmed model...

Optimizing original model:
Original model objective value: 0.084444

Optimizing trimmed model:
Trimmed model objective value: 0.084444

Difference in objective values: 0.000000e+00

Success! Trimmed model produces similar results

- N reactions in original model: 7786
- N reactions in trimmed model:  4786

- N metabolites in original model: 5324
- N metabolites in trimmed model:  2570

- N genes in original model: 1673
- N genes in trimmed model:  1412


Metabolite,Reaction,Flux,C-Number,C-Flux
ala_L_e,EX_ala_L_e,0.005845,3,0.35%
arg_L_e,EX_arg_L_e,0.05,6,6.04%
asn_L_e,EX_asn_L_e,0.0236,4,1.90%
asp_L_e,EX_asp_L_e,0.03131,4,2.52%
chol_e,EX_chol_e,0.005,5,0.50%
glc_D_e,EX_glc_e,0.1969,6,23.80%
gln_L_e,EX_gln_L_e,0.05631,5,5.67%
glu_L_e,EX_glu_L_e,0.05,5,5.04%
gthrd_e,EX_gthrd_e,0.05,10,10.07%
his_L_e,EX_his_L_e,0.05,6,6.04%


In [30]:
print(f"Writing final trimmed model to {final_model_path}")

# Clear notes to reduce file size (following the pattern from other notebooks)
trimmed_model.notes = {}
for m in trimmed_model.metabolites:
    m.notes = {}

write_sbml_model(trimmed_model, final_model_path)
print("Model successfully written!")
print(f"Final model saved to: {final_model_path}")

Writing final trimmed model to models/Recon2.2/Recon2.2_FBC_trimmed_final.xml
Model successfully written!
Final model saved to: models/Recon2.2/Recon2.2_FBC_trimmed_final.xml
